# 3. Costed fire stations, written with sets and dictionaries

Lecture: *Week 6 - Integer Linear Programming - Further Problems*.

*Original script: `5 Source Files/Python/costed_firestation_sets_dictionaries.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Define the binary variables
model = ConcreteModel()

firestations = ['A','B','C','D']
cities = [1,2,3,4,5]

In [ ]:
# variable to represent if fire station is selected
model.x = Var(firestations, within=Binary)

cost = {'A': 12, 'B': 7, 'C': 10, 'D': 5}
covers = {
    'A': [1,3],
    'B': [2,3,4],
    'C': [2,5],
    'D': [5]
    }

In [ ]:
# Create the optimization problem
model.obj = Objective(expr=sum(cost[station] * model.x[station] for station in firestations), sense=minimize)

def cover_constraint(model, city):
    return sum(model.x[s] for s in firestations if city in covers[s]) >= 1

model.cover_constaint = Constraint(cities, rule=cover_constraint)

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

In [ ]:
# Display the model
model.pprint()

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

In [ ]:
# Display the results
model.display()